# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

---

## Setup

In [11]:
import asyncio
import json
import os
import time
from pathlib import Path

import pandas as pd
from openai import AsyncOpenAI

# Make sure your OPENAI_API_KEY is set in the environment
# assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first'


client = AsyncOpenAI()

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

Setup complete.


## Step 1 — Load the data

In [12]:
DATA_DIR = Path('../data')   # adjust if your folder layout differs

snippets = [json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip()]
golden = {row['id']: row for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}

print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries.')
print('Sample snippet:', snippets[0])

Loaded 10 snippets, 10 golden entries.
Sample snippet: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}


## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [13]:
def prompt_zero_shot(snippet_text: str) -> list[dict]:
    """Strategy 1 — zero-shot. Just ask, no examples, no persona."""

    return [
        {
            "role": "user",
            "content": f"""
Extract the following information from the job posting:

1. company: Company or organization name.
2. role: Job title.
3. years_experience_required: Minimum years of experience required.

Rules:
- Return years_experience_required as an integer.
- If no experience is required, return 0.
- If experience is not mentioned, return null.
- If a range is provided (e.g., 3 to 5 years), return the minimum.
- Return only valid JSON.
- Do not include explanations.

Job posting:
{snippet_text}

Expected JSON format:
{{
    "company": "string or null",
    "role": "string or null",
    "years_experience_required": "integer or null"
}}
"""
        }
    ]


def prompt_few_shot(snippet_text: str) -> list[dict]:
    """Strategy 2 — few-shot. Include 2-3 worked examples in the prompt."""

    prompt = f"""
You are extracting information from job postings.

Extract these three fields:
- company: Company name
- role: Job title
- years_experience_required: Minimum required years of experience as an integer.

Rules:
1. Return 0 if no prior experience is required or fresh graduates are welcome.
2. Return null if the posting does not specify an experience requirement.
3. If a range is provided (e.g., 3-5 years), return the minimum value.
4. Return only valid JSON. Do not include explanations.

Here are some examples:

Example 1:
Job posting:
"Acme Corp is hiring a Senior Software Engineer.
Candidates should have 5+ years of backend development experience."

Expected output:
{{
    "company": "Acme Corp",
    "role": "Senior Software Engineer",
    "years_experience_required": 5
}}

Example 2:
Job posting:
"Hooli is looking for a Junior Frontend Developer.
Fresh graduates are welcome. No prior experience is required."

Expected output:
{{
    "company": "Hooli",
    "role": "Junior Frontend Developer",
    "years_experience_required": 0
}}

Example 3:
Job posting:
"Cyberdyne Systems is hiring an AI/ML Research Scientist.
No specific years of experience requirement is listed."

Expected output:
{{
    "company": "Cyberdyne Systems",
    "role": "AI/ML Research Scientist",
    "years_experience_required": null
}}

Now extract information from this job posting:

Job posting:
{snippet_text}

Expected JSON output:
"""

    return [
        {
            "role": "user",
            "content": prompt
        }
    ]


def prompt_structured(snippet_text: str) -> list[dict]:
    """Strategy 3 — structured / role-based. Use a system prompt with a persona and explicit JSON schema."""

    system_prompt = """
You are an expert technical recruiter and job description analyst.

Your task is to extract structured information from job postings.

Extract exactly these three fields:

1. company:
   - The organization or company name.
   - Return a string.
   - Return null if not mentioned.

2. role:
   - The job title or position.
   - Return a string.
   - Return null if not mentioned.

3. years_experience_required:
   - Return the minimum required years of experience as an integer.
   - If a range is specified (e.g., 3-5 years), return 3.
   - If fresh graduates are welcome or no prior experience is required, return 0.
   - If no specific experience requirement is mentioned, return null.

Output requirements:
- Return only valid JSON.
- Use exactly the three field names specified above.
- Do not add extra fields or explanations.
- Follow this JSON structure:

{
    "company": "string or null",
    "role": "string or null",
    "years_experience_required": "integer or null"
}
"""

    user_prompt = f"""
Extract the required information from the following job posting:

{snippet_text}
"""

    return [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]


def prompt_cot(snippet_text: str) -> list[dict]:
    """Strategy 4 — chain-of-thought. Ask the model to reason before answering."""

    system_prompt = """
You are an expert job description analyst.

Analyze the job posting carefully before extracting the information.

Follow these steps internally:
1. Identify the company name.
2. Identify the job title.
3. Find the experience requirement.
4. Determine the minimum required years of experience.
5. Check whether experience is optional, not required, or unspecified.

Extraction rules:
- Return the minimum value for experience ranges (e.g., 3-5 years means 3).
- Return 0 if no prior experience is required.
- Return null if the experience requirement is not specified.
- Return null for company or role if they are not mentioned.

Return only valid JSON with exactly these fields:
{
    "company": "string or null",
    "role": "string or null",
    "years_experience_required": "integer or null"
}

Do not include your internal reasoning or explanations in the output.
"""

    user_prompt = f"""
Analyze this job posting and return the extracted information:

{snippet_text}
"""

    return [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]


STRATEGIES = {
    'zero_shot': prompt_zero_shot,
    'few_shot': prompt_few_shot,
    'structured': prompt_structured,
    'cot': prompt_cot,
}

## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [14]:
def parse_response(text: str) -> dict | None:
    """Parse JSON from model response. Return None if invalid."""

    try:
        text = text.replace("```json", "").replace("```", "").strip()
        result = json.loads(text)

        if isinstance(result, dict):
            return result

        return None

    except json.JSONDecodeError:
        return None


import time

async def run_one(strategy_name: str, snippet: dict) -> dict:
    """Run one strategy on one snippet."""

    start_time = time.time()

    messages = STRATEGIES[strategy_name](snippet["snippet"])

    # Call the model
    response = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        response_format={"type": "json_object"},
        temperature=TEMPERATURE,
    )

    # Calculate execution time
    elapsed_time = time.time() - start_time

    # Parse model response
    result = parse_response(response.choices[0].message.content)

    # Calculate token usage
    prompt_tokens = response.usage.prompt_tokens
    completion_tokens = response.usage.completion_tokens

    # Example GPT-4o-mini pricing per 1 million tokens
    cost = (
        prompt_tokens * 0.15 / 1_000_000
        + completion_tokens * 0.60 / 1_000_000
    )

    return {
        "strategy": strategy_name,
        "snippet_id": snippet.get("id"),
        "company": result.get("company") if result else None,
        "role": result.get("role") if result else None,
        "years_experience_required": result.get("years_experience_required") if result else None,
        "latency_seconds": round(elapsed_time, 3),
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "cost_usd": round(cost, 8),
        "success": result is not None
    }




In [15]:
import asyncio

async def run_all() -> list[dict]:
    """Run all 10 × 4 = 40 calls in parallel."""

   
    tasks = [
        run_one(strategy, snippet)
        for snippet in snippets
        for strategy in STRATEGIES.keys()
    ]

    results = await asyncio.gather(*tasks)

    return results

In [16]:
# Run it
results = await run_all()
print(f'Got {len(results)} results.')
results[0]

Got 40 results.


{'strategy': 'zero_shot',
 'snippet_id': 'j01',
 'company': 'Acme Corp',
 'role': 'Senior Software Engineer',
 'years_experience_required': 5,
 'latency_seconds': 1.676,
 'prompt_tokens': 187,
 'completion_tokens': 30,
 'cost_usd': 4.605e-05,
 'success': True}

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [17]:
RUBRIC = """Evaluate the answer on three dimensions. For each, give a score 1-4:

ACCURACY (1-4)
  4 = fully correct given the corpus
  3 = mostly correct, minor issues
  2 = partially correct, significant issues
  1 = incorrect or misleading

GROUNDEDNESS (1-4)
  4 = every claim traces to the corpus
  3 = mostly grounded, small unsupported additions
  2 = mix of grounded and invented content
  1 = substantially invented or hallucinated

FORMAT (1-4)
  4 = clear, appropriately concise, well-structured
  3 = clear but slightly verbose or terse
  2 = confusing structure or wrong length
  1 = badly formatted, hard to read"""

print(RUBRIC)


Evaluate the answer on three dimensions. For each, give a score 1-4:

ACCURACY (1-4)
  4 = fully correct given the corpus
  3 = mostly correct, minor issues
  2 = partially correct, significant issues
  1 = incorrect or misleading

GROUNDEDNESS (1-4)
  4 = every claim traces to the corpus
  3 = mostly grounded, small unsupported additions
  2 = mix of grounded and invented content
  1 = substantially invented or hallucinated

FORMAT (1-4)
  4 = clear, appropriately concise, well-structured
  3 = clear but slightly verbose or terse
  2 = confusing structure or wrong length
  1 = badly formatted, hard to read


In [18]:
def normalise_text(value):
    """Normalise a string for case-insensitive, whitespace-trimmed comparison."""
    return value.strip().casefold() if isinstance(value, str) else None


def normalise_years(value):
    """Convert integer-like model output (including 5+) to an integer."""
    if value is None:
        return None
    if isinstance(value, bool):
        return None
    if isinstance(value, int):
        return value
    if isinstance(value, float):
        return int(value) if value.is_integer() else None
    if isinstance(value, str):
        match = re.search(r"\d+", value)
        return int(match.group()) if match else None
    return None


def score_accuracy(extracted: dict | None, gold: dict) -> int:
    """Compare all three fields and return 0-3 exact matches."""
    if not isinstance(extracted, dict):
        return 0

    score = 0
    for field in ("company", "role"):
        if normalise_text(extracted.get(field)) == normalise_text(gold.get(field)):
            score += 1

    extracted_years = normalise_years(extracted.get("years_experience_required"))
    gold_years = normalise_years(gold.get("years_experience_required"))
    if extracted_years == gold_years:
        score += 1
    return score


async def score_llm_judge(
    snippet_text: str,
    extracted: dict | None,
    gold: dict,
) -> int:
    """Use gpt-4o with a temperature of zero to judge extraction quality."""
    judge_messages = [
        {
            "role": "system",
            "content": (
                "You are an exact scoring judge for structured job-post extraction. "
                "Return only a JSON object like {\"score\": 4}. "
                "Score 4 only if all three fields are correct, 3 if exactly two are "
                "correct, 2 if exactly one is correct, and 1 if none are correct."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Job posting:\n{snippet_text}\n\n"
                f"Gold extraction:\n{json.dumps(gold, ensure_ascii=False)}\n\n"
                f"Model extraction:\n{json.dumps(extracted, ensure_ascii=False)}"
            ),
        },
    ]

    response = await client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=judge_messages,
        response_format={"type": "json_object"},
        temperature=TEMPERATURE,
    )
    judged = parse_response(response.choices[0].message.content or "")
    score = judged.get("score") if judged else None
    if isinstance(score, int) and 1 <= score <= 4:
        return score
    return max(1, score_accuracy(extracted, gold) + 1)


In [19]:

async def apply_scoring(results: list[dict]) -> list[dict]:
    """Attach strict accuracy and LLM-judge scores to every result."""

    scored = []

    for result in results:

        # Reconstruct extracted data from the flattened result structure
        extracted = result.get("parsed")

        if extracted is None:
            extracted = {
                "company": result.get("company"),
                "role": result.get("role"),
                "years_experience_required": result.get(
                    "years_experience_required"
                ),
            }

        # Retrieve ground truth
        gold = golden[result["snippet_id"]]

        # Retrieve original job posting
        snippet = next(
            row for row in snippets
            if row["id"] == result["snippet_id"]
        )

        # Calculate strict accuracy (0-3)
        accuracy = score_accuracy(extracted, gold)

        # Calculate LLM judge score (1-4)
        judge_score = await score_llm_judge(
            snippet["snippet"],
            extracted,
            gold
        )

        # Preserve original result and attach scoring metrics
        row = dict(result)
        row["accuracy"] = accuracy
        row["parse_success"] = int(
            isinstance(extracted, dict)
            and all(extracted.get(k) is not None for k in (
                "company",
                "role",
                "years_experience_required"
            ))
        )
        row["llm_judge_score"] = judge_score

        scored.append(row)

    return scored


# Execute scoring
scored = await apply_scoring(results)

print(f"Scored {len(scored)} results.")



Scored 40 results.


## Step 5 — Build the comparison table

In [20]:
df = pd.DataFrame(scored)

summary = df.groupby("strategy").agg({
    "accuracy": "mean",
    "parse_success": "mean",
    "llm_judge_score": "mean",
    "cost_usd": "sum",
    "latency_seconds": "median",
}).round(3)

summary.columns = [
    "Accuracy (mean of 3)",
    "Parse rate",
    "Judge score",
    "Total cost ($)",
    "Latency p50 (s)",
]
summary


,Accuracy (mean of 3),Parse rate,Judge score,Total cost ($),Latency p50 (s)
strategy,,,,,
cot,2.8,0.9,4.0,0.001,1.707
few_shot,3.0,0.9,4.0,0.001,1.894
structured,2.8,0.9,4.0,0.001,1.609
zero_shot,2.9,0.9,4.0,0.000,1.746


## Step 6 — Write your reflection

Open `mp1_writeup.md` and answer the four questions from the brief.

Then commit:

```bash
git add mp1/
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```